In [0]:
%pip install -q pandas transformers torch tqdm "optimum[onnxruntime]" onnxruntime

Lyrics Emotion Classification Pipeline — GoEmotions (SamLowe/roberta-base-go_emotions-onnx)
=============================================================================================
Input:  analysis_df with columns: rank, artist, title, region,
        spotify_uri, lyrics_in_en, original_lang
Output: analysis_df + 28 emotion score columns + dominant_emotion,
        dominant_score, low_confidence

Supervised, fine-tuned multi-label classifier (28 GoEmotions labels) run via
ONNX Runtime through `optimum`. Faster single-pass inference than the
zero-shot NLI approach, but locked into GoEmotions' general-purpose label
set (Reddit-comment tone) rather than the custom song-emotion vocabulary
used in 04.1. See 04.1_classification_zeroshot.ipynb for that variant.

Shared scoring contract
-----------------------
04.1 and 04.2 use different models and different label sets, but they are
deliberately held to the **same scoring contract** so their outputs can be
compared in 07_compare_classifiers.ipynb:

1. **Independent per-label probabilities in [0, 1].** Neither notebook applies
   a softmax across labels, so scores do not sum to 1 and a song can score high
   on several emotions at once. Here that is the model's native sigmoid head;
   in 04.1 it is `multi_label=True`.
2. **`unclassified` means "no scoreable lyrics"** (every score is 0) — never
   "low confidence". Both forks therefore drop the same ~107 empty-lyric songs
   and carry the same song set forward.
3. **Confidence is data, not a filter.** `dominant_score` records the winning
   score and `low_confidence` flags it against the shared `MIN_CONFIDENCE`
   constant. Downstream notebooks can filter on it — identically across forks —
   instead of each fork silently dropping a different number of rows.

Point 3 replaces this notebook's old `UNCLASSIFIED_THRESHOLD = 0.30` hard drop.
That threshold, versus 04.1's `> 0`, accounted for essentially the entire
apparent coverage gap between the two classifiers (201 vs 108 songs dropped) —
at a common bar both discard the same ~107 songs, all of them empty lyrics.
RoBERTa's sigmoids are calibrated lower than bart-mnli's entailment
probabilities, so a fixed absolute bar was never neutral between them.

One asymmetry remains and cannot be config'd away: GoEmotions has a `neutral`
label and 04.1's taxonomy has no equivalent. It is handled downstream in 05.2
(`dominant_emotion_emotive`) rather than here.

In [0]:
import re
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root

# Load from 03_lyrics_trans (emotional analysis ready - all lyrics translated to English)
analysis_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "03_lyrics_trans.csv")
# DATABRICKS PATH
# analysis_df = pd.read_csv(Path("/Volumes/songs_db/default/storage/03_lyrics_trans.csv"))

if "lyrics_in_en" not in analysis_df.columns:
    raise KeyError("Expected 'lyrics_in_en' column in 03_lyrics_trans.csv")

print(f'Loaded {len(analysis_df)} songs')
print(f'Columns: {list(analysis_df.columns)}')

display(analysis_df.head(3))

In [ ]:
# Config

MODEL_ID = "SamLowe/roberta-base-go_emotions-onnx"

# GoEmotions' fixed 28-label taxonomy (27 emotions + neutral).
# Unlike 04.1's EMOTIONS list, this set is not customizable without
# fine-tuning — it's whatever the model was trained on.
EMOTIONS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral",
]

# How many words per chunk (RoBERTa-base context is 512 tokens; stay well under it)
CHUNK_WORDS = 350

# ── Shared with 04.1 — keep these two in sync ────────────────────────────────
# Confidence bar for the `low_confidence` FLAG. It does not drop any rows; see
# the scoring contract above. Both notebooks must use the same value or the
# flag stops meaning the same thing in each fork.
#
# Replaces the old UNCLASSIFIED_THRESHOLD, which dropped ~94 songs here that
# 04.1 kept — a config difference that masqueraded as a coverage difference
# between the classifiers.
MIN_CONFIDENCE = 0.30
# ─────────────────────────────────────────────────────────────────────────────

# Checkpointing
checkpoint_every = 10
checkpoint_path = PROJECT_ROOT / "data" / "processed" / "04.2_emotion_scores_goemotions_checkpoint.csv"
# DATABRICKS PATH
# checkpoint_path = Path("/Volumes/songs_db/default/storage/04.2_emotion_scores_goemotions_checkpoint.csv")

### Clean lyrics

In [0]:
# ─────────────────────────────────────────────
# 1. CLEANING  — keep punctuation & case
# ─────────────────────────────────────────────
# Identical to 04.1 — cleaning is model-agnostic.

def clean_lyrics(text: str, artist: str = '') -> str:
    """
    Remove structural noise but preserve punctuation and casing.
    Punctuation (! ? ...) and capitalisation carry emotional signal
    for the classifier — don't strip them.

    artist: the artist string from the dataframe row. When provided,
    the first line is dropped only if its tokens are a subset of the
    known artist names — much more precise than a regex heuristic.
    Falls back to the regex heuristic when artist is unavailable.
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    def is_artist_credit(line: str) -> bool:
        """
        True if every name token in `line` exists in the artist pool.
        Both strings are split on commas, ampersands, and feat/ft.

            artist = "Jason, Bonnie"          line = "Jason"          → True
            artist = "ARIA VEGA, Ryan Castro" line = "Ryan Castro"    → True
            artist = "Jason, Bonnie"          line = "Baby come back" → False
        """
        splitter = r'[,&]|\bfeat\.?\b|\bft\.?\b'
        artist_tokens = {
            t.strip().lower()
            for t in re.split(splitter, artist, flags=re.IGNORECASE)
            if t.strip()
        }
        line_tokens = {
            t.strip().lower()
            for t in re.split(splitter, line, flags=re.IGNORECASE)
            if t.strip()
        }
        return bool(line_tokens) and line_tokens.issubset(artist_tokens)

    # Remove section headers: [Verse 1], [Chorus], [Bridge] etc.
    text = re.sub(r'\[[^\]]*\]', '', text)

    # Remove repetition annotations: (x3), (×2), (2x)
    text = re.sub(r'\([\d×xX]+\)', '', text)

    # Remove leading artist/feature credits that sometimes appear
    # at the top of translated lyrics (e.g. "ARIA VEGA, Ryan Castro\n").
    lines = text.strip().splitlines()
    if lines:
        first = lines[0].strip()
        if artist and is_artist_credit(first):
            lines = lines[1:]
        elif not artist and re.match(r'^[A-Za-z\s,&]+$', first) and len(first) < 80:
            lines = lines[1:]
    text = '\n'.join(lines)

    # Collapse excessive blank lines but keep single line breaks
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()


def dedupe_lines(text: str) -> str:
    """
    Remove duplicate lines (repeated choruses inflate scores).
    Keeps first occurrence; preserves order.
    """
    seen = set()
    result = []
    for line in text.splitlines():
        key = line.strip().lower()
        if key and key not in seen:
            seen.add(key)
            result.append(line.strip())
    return ' '.join(result)


def prepare_lyrics(text: str, artist: str = '') -> str:
    return dedupe_lines(clean_lyrics(text, artist=artist))

### Chunking

In [0]:
# ─────────────────────────────────────────────
# 2. CHUNKING  — handle long lyrics gracefully
# ─────────────────────────────────────────────
# Identical to 04.1.

def chunk_text(text: str, chunk_words: int = CHUNK_WORDS) -> list[str]:
    """Split text into word-count chunks with a small overlap."""
    words = text.split()
    if not words:
        return []
    overlap = 30
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_words
        chunks.append(' '.join(words[start:end]))
        start = end - overlap  # small overlap so context isn't lost at boundaries
        if start >= len(words):
            break
    return chunks

### Classification

In [ ]:
# ─────────────────────────────────────────────
# 3. CLASSIFICATION
# ─────────────────────────────────────────────

def load_classifier():
    """
    SamLowe/roberta-base-go_emotions-onnx — RoBERTa fine-tuned on GoEmotions,
    exported to ONNX and served via optimum's ONNX Runtime backend.
    Single supervised forward pass per input (no NLI hypothesis pairs), so
    this is substantially faster than the zero-shot classifier in 04.1.

    top_k=None returns scores for all 28 labels per input (not just the
    top prediction). function_to_apply='sigmoid' matches how the model was
    trained: multi-label, independent per-label probabilities. 04.1 is set to
    multi_label=True for the same reason — both forks emit independent
    per-label scores in [0, 1], which is the shared scoring contract that makes
    them comparable at all.
    """
    from optimum.onnxruntime import ORTModelForSequenceClassification
    from transformers import AutoTokenizer, pipeline

    print(f"Loading GoEmotions ONNX classifier ({MODEL_ID})...")
    model = ORTModelForSequenceClassification.from_pretrained(MODEL_ID)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    return pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        top_k=None,
        function_to_apply="sigmoid",
    )


def classify_song(lyrics: str, classifier) -> dict:
    """
    Classify a single song's lyrics.

    Each chunk gets independent sigmoid scores for all 28 labels — a song can
    score high on several emotions at once, and scores do not sum to 1. 04.1
    behaves the same way; the two differ in model and label set, not in scoring
    mechanics. Chunk scores are averaged to produce a song-level score.

    Note that RoBERTa's sigmoids sit systematically lower than bart-mnli's
    entailment probabilities (median top score ~0.50 vs ~0.97), so raw
    magnitudes are still not interchangeable between the forks even under the
    shared contract. Compare within a fork, or z-score across regions first —
    see 06.1 / 06.2 § 4 and 07_compare_classifiers.

    Chunks are classified one at a time, NOT batched. This ONNX export of
    roberta-base-go_emotions has a fixed-batch assumption in its position-
    embedding broadcast — passing a list of >1 texts triggers
    "INVALID_ARGUMENT ... Expand node ... invalid expand shape" from
    onnxruntime. Single-example calls avoid the broadcast entirely.

    Whether a single-string call returns a flat list of 28 dicts or a
    nested [[...]] list varies by transformers/optimum version, so unwrap
    defensively rather than assume one shape.

    Returns dict {emotion: score}.
    """
    if not isinstance(lyrics, str) or not lyrics.strip():
        return {e: 0.0 for e in EMOTIONS}

    chunks = chunk_text(lyrics)
    all_scores = []

    for chunk in chunks:
        result = classifier(chunk, truncation=True)
        if isinstance(result, list) and result and isinstance(result[0], list):
            result = result[0]  # unwrap [[{...}, ...]] -> [{...}, ...]
        all_scores.append({d["label"]: d["score"] for d in result})

    avg = {
        e: round(sum(s.get(e, 0.0) for s in all_scores) / len(all_scores), 4)
        for e in EMOTIONS
    }
    return avg


def load_checkpoint(checkpoint_path: Path) -> pd.DataFrame | None:
    """
    Load an existing checkpoint CSV if it exists, else return None.

    NOTE: a COMPLETE checkpoint makes this notebook a no-op — Step 3 below only
    classifies rows with NaN scores, so changing a scoring setting and
    re-running will silently reuse the old scores. Delete the checkpoint when
    you change how scores are produced. (Changing only MIN_CONFIDENCE is safe:
    it is applied in Step 4, which always re-runs.)
    """
    if checkpoint_path.exists():
        df = pd.read_csv(checkpoint_path)
        emotion_cols = [f"emotion_{e}" for e in EMOTIONS]
        already_done = df[emotion_cols].notna().all(axis=1).sum()
        print(f"Checkpoint found: {already_done} / {len(df)} songs already classified.")
        if already_done == len(df):
            print("  ⚠ Checkpoint is COMPLETE — no song will be re-scored this run.")
            print("    Delete it first if you changed EMOTIONS, the model, or the")
            print("    chunking. Step 4 (dominant/confidence) always re-derives.")
        return df
    return None


def save_checkpoint(df: pd.DataFrame, checkpoint_path: Path) -> None:
    """Write the current state of df (including any NaN emotion cols) to disk."""
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(checkpoint_path, index=False)

### Main Pipeline

In [ ]:
# ── Step 2.5: prepare dataframe state for classification ─────────────────────
emotion_cols = [f"emotion_{e}" for e in EMOTIONS]
df = load_checkpoint(checkpoint_path)

if df is None:
    df = analysis_df.copy()
    for col in emotion_cols:
        if col not in df.columns:
            df[col] = pd.NA

if 'lyrics_prepared' not in df.columns:
    df['lyrics_prepared'] = df.apply(
        lambda r: prepare_lyrics(r.get('lyrics_in_en', ''), artist=r.get('artist', '')),
        axis=1
    )

needs_classification = df[emotion_cols].isna().any(axis=1)
todo = df[needs_classification].index.tolist()
print(f"Songs pending classification: {len(todo)} / {len(df)}")

# ── Step 3: classify only missing rows + checkpointing ─────────────────────
if len(todo) == 0:
    print("No unclassified rows. Using existing scores.")
else:
    classifier = load_classifier()
    n_saved = 0

    for i, idx in enumerate(tqdm(todo, desc="Classifying songs", unit="song")):
        lyrics = df.at[idx, 'lyrics_prepared']
        scores = classify_song(lyrics, classifier)

        for emotion, score in scores.items():
            df.at[idx, f"emotion_{emotion}"] = score

        # Flush to disk every checkpoint_every songs
        n_saved += 1
        if n_saved % checkpoint_every == 0:
            save_checkpoint(df, checkpoint_path)
            tqdm.write(f"  ✓ Checkpoint saved ({n_saved} songs classified this run)")

    # Final save after the loop completes
    save_checkpoint(df, checkpoint_path)
    print(f"Done. Checkpoint saved to {checkpoint_path}")

# ── Step 4: derive dominant emotion ──────────────────────────────────────────
# Shared scoring contract — this block is intentionally IDENTICAL in 04.1.
#
# Scores are independent per-label probabilities, so the "dominant" emotion is
# simply the argmax; there is no competition to resolve. `unclassified` is
# reserved for songs with NO scoreable lyrics (every score 0). Low confidence is
# recorded and flagged, NOT dropped — the previous `>= UNCLASSIFIED_THRESHOLD`
# drop is what left this fork holding ~94 fewer songs than 04.1 and made a
# threshold choice look like a classifier weakness.
df[emotion_cols] = df[emotion_cols].astype(float)

max_score = df[emotion_cols].max(axis=1)
has_signal = max_score > 0

df['dominant_emotion'] = (
    df[emotion_cols].idxmax(axis=1).str.replace('emotion_', '', regex=False)
       .where(has_signal, other='unclassified')
)
df['dominant_score'] = max_score.where(has_signal)
df['low_confidence'] = has_signal & (max_score < MIN_CONFIDENCE)

print(f"unclassified (no scoreable lyrics): {(~has_signal).sum()} / {len(df)}")
print(f"low_confidence (scored, but < {MIN_CONFIDENCE}): {df['low_confidence'].sum()} / {len(df)}")

df = df.drop(columns=['lyrics_prepared'])

# Output to 04.2_emotion_scores_goemotions.csv with appropriate schema
final_output_path = PROJECT_ROOT / "data" / "processed" / "04.2_emotion_scores_goemotions.csv"
df.to_csv(final_output_path, index=False)
print(f"Saved emotion analysis to {final_output_path}")
display(df.head(3))

# 29m 7s runtime.

In [ ]:
def regional_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Average emotion scores per region.
    Useful for radar/heatmap visualisation.
    """
    titles_path = PROJECT_ROOT / "data" / "processed" / "00_titles.csv"
    titles_df = pd.read_csv(titles_path)[["spotify_uri", "region"]]

    emotion_cols = [f"emotion_{e}" for e in EMOTIONS]
    merged = titles_df.merge(df[["spotify_uri"] + emotion_cols], on="spotify_uri", how="inner")

    summary = merged.groupby('region')[emotion_cols].mean().round(3)
    summary.columns = [c.replace('emotion_', '') for c in summary.columns]
    return summary

summary = regional_summary(df)
display(summary)
summary.to_csv(PROJECT_ROOT / "data" / "processed" / "04.2_regional_summary_goemotions.csv", index=False)
# DATABRICKS PATH
# summary.to_csv(Path("/Volumes/songs_db/default/storage/04.2_regional_summary_goemotions.csv"), index=False)